# Lab 4: Linaro Forge MAP Walkthrough
## Parse, Visualize, and Optimize HPC Profiles

**Learning objective:** Parse synthetic MAP profile data, render a flamechart-equivalent visualization, and identify optimization opportunities.

**Note:** Linaro Forge MAP is commercial software (2-week trial available). This lab uses synthetic profile data shaped like real MAP output to teach interpretation. (facts)

In [ ]:
import json
import sys
import platform
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.insert(0, str(Path.cwd()))
from code.utils.map_profiler import (
    parse_map_file,
    generate_flamechart_data,
    analyze_profile_for_optimization,
)
from code.utils.config import ISA_COLORS, PLOTLY_DARK_TEMPLATE

print(f"Python {platform.python_version()}")
print(f"Environment: {platform.system()} {platform.release()}")
print(f"Imports: OK")

## What is Linaro Forge MAP?

**History:**
- Allinea Software created MAP in 2005 as a statistical sampler for HPC applications.
- Arm acquired Allinea in December 2016. (facts)
- Linaro acquired Arm Forge (including MAP) on January 30, 2023. (facts)

**Current version (April 2026):** Linaro Forge 25.1.2 (facts)

**License:** Commercial. 2-week free trial available. (facts)

**Technology:** Adaptive statistical sampling — typical 5% runtime overhead. (facts)

**Metrics captured:** CPU time, MPI time, I/O time, thread activity, memory usage. (facts)

**Output format:** Binary `.map` file (appname_#p_YYYY-MM-DD_HH-MM.map). (facts)

**Scalability:** Supports up to 2048 MPI tasks. (facts)

## MAP vs perf vs gprof — What MAP Uniquely Shows

| Feature | MAP | perf | gprof |
|---------|-----|------|-------|
| **Source-level attribution** | Yes — line-level hotspots | Limited — kernel symbols | Yes — function-level |
| **MPI-awareness** | Yes — MPI rank profiling, collective detection | No | No |
| **Overhead** | ~5% | <2% (PMU-based) | >20% (instrumentation) |
| **Output format** | Binary .map + GUI + perf-report | stdout/text | gmon.out (binary) |
| **Thread activity** | Yes — per-thread, OpenMP detection | Yes — sampling | Limited |
| **I/O profiling** | Yes — MPI-IO, file syscalls | Limited | No |
| **Commercial** | Yes (facts) | Free (Linux) | Free (Unix) |

**Why use MAP:**
- Only tool combining **MPI-aware profiling + source attribution + low overhead** (facts)
- Critical for debugging weak-scaling issues in HPC codes (reasoned)
- Identifies collective communication bottlenecks that perf/gprof miss (reasoned)

## Constants and Configuration

In [ ]:
MOCK_PROFILE_PATH = Path("code/data/mock_map_profile.json")

COLOR_MAP_CATEGORY = {
    "compute": ISA_COLORS["SVE2"],  # #00CC66 green
    "mpi": ISA_COLORS["NEON"],      # #0066CC blue
    "io": ISA_COLORS["SME2"],       # #FFAA00 amber
    "other": "#CCCCCC",              # gray
}

print("Configuration loaded.")
print(f"ISA_COLORS: {ISA_COLORS}")
print(f"PLOTLY_DARK_TEMPLATE keys: {list(PLOTLY_DARK_TEMPLATE.keys())}")

## Load and Parse Mock MAP Profile

In [ ]:
with open(MOCK_PROFILE_PATH) as f:
    raw_data = json.load(f)

profile = parse_map_file(raw_data)

print("=== MAP Profile Summary ===")
print(f"Application: {raw_data['metadata']['app']}")
print(f"Tool version: {raw_data['metadata']['tool']}")
print(f"Ranks: {profile['ranks']}")
print(f"Total runtime: {profile['total_runtime_s']:.1f} seconds")
print(f"Hotspots sampled: {profile['samples']}")
print(f"\nTime breakdown:")
print(f"  CPU: {profile['cpu_time_pct']:.1f}%")
print(f"  MPI: {profile['mpi_time_pct']:.1f}%")
print(f"  I/O: {profile['io_time_pct']:.1f}%")
print(f"  Thread overhead: {profile['thread_overhead_pct']:.1f}%")

## Reading the Time Breakdown

The time breakdown shows where the application spent its runtime across four categories:

- **CPU:** Useful computation on the processor. High CPU % is good; indicates the app is computing rather than waiting.
- **MPI:** Time spent in MPI collective operations (Allreduce, Bcast, etc.) or MPI point-to-point communication.
- **I/O:** Time in file I/O (read/write), including MPI-IO and POSIX syscalls.
- **Thread overhead:** Synchronization, lock contention, and OpenMP overhead.

**Ideal target:** 70–90% CPU, <20% MPI, <5% I/O, <5% thread overhead. (reasoned)

## Visualization: Time Breakdown Pie Chart

In [ ]:
labels = ["CPU", "MPI", "I/O", "Thread Overhead"]
values = [
    profile["cpu_time_pct"],
    profile["mpi_time_pct"],
    profile["io_time_pct"],
    profile["thread_overhead_pct"],
]
colors = [
    COLOR_MAP_CATEGORY["compute"],
    COLOR_MAP_CATEGORY["mpi"],
    COLOR_MAP_CATEGORY["io"],
    COLOR_MAP_CATEGORY["other"],
]

fig = go.Figure(
    data=[go.Pie(labels=labels, values=values, marker=dict(colors=colors))]
)

fig.update_layout(
    title="Time Breakdown (8-rank HPL benchmark, 47.3 seconds runtime)",
    font=dict(color="white", size=12),
    paper_bgcolor="#000000",
    plot_bgcolor="#111111",
    hovermode="closest",
)

fig.show()

## Hotspot Table — Finding the Bottleneck

The hotspots table lists the functions where the profiler's statistical sampler found the most time spent. Each row shows:

- **Function:** Function name from the binary symbol table.
- **File:** Source file where the function is defined (if debug symbols present).
- **Line:** Source line of the function entry (approximate; sampling is not exact).
- **CPU%:** Percentage of total runtime spent in this function.
- **MPI%:** Percentage of total runtime spent in MPI calls from within this function.

**How to read:** The top rows are the biggest consumers of time. Optimize the top 3–5 for maximum impact. (reasoned)

In [ ]:
hotspots_df = pd.DataFrame(profile["hotspots"])
hotspots_df = hotspots_df.sort_values("pct_cpu_time", ascending=False)
hotspots_df = hotspots_df.reset_index(drop=True)

display_df = hotspots_df[[
    "function_name",
    "file",
    "line",
    "pct_cpu_time",
    "pct_mpi_time",
]].copy()
display_df.columns = ["Function", "File", "Line", "CPU%", "MPI%"]

def highlight_cpu(val):
    """Color cells by CPU% value: higher = greener."""
    if val > 25:
        color = "#00CC66"
    elif val > 15:
        color = "#00AA44"
    elif val > 5:
        color = "#008833"
    else:
        color = "#555555"
    return f"background-color: {color}; color: white;"

styled_df = display_df.style.applymap(
    highlight_cpu, subset=["CPU%"]
)

display(styled_df)

## Flamechart-Equivalent Visualization

A flamechart shows the call stack depth over time. Since we're working with aggregated hotspot data, we render an equivalent horizontal bar chart where:

- **X-axis:** Percentage of total runtime (0–100%).
- **Y-axis:** Hotspot rank (top contributor = y=0).
- **Bar width:** CPU time for that hotspot.
- **Color:** Compute (green), MPI (blue), I/O (amber), other (gray).

This layout mimics MAP's graphical profile view and is commonly called the "flamechart equivalent" in text-based or programmatic profiling tools. (reasoned)

In [ ]:
flamechart_data = generate_flamechart_data(profile)

fig = go.Figure()

for entry in flamechart_data:
    fig.add_trace(
        go.Bar(
            x=[entry["pct"]],
            y=[entry["y_level"]],
            name=entry["label"],
            orientation="h",
            marker=dict(color=entry["color"]),
            hovertemplate=(
                f"<b>{entry['label']}</b><br>"
                f"CPU time: {entry['pct']:.1f}%<br>"
                f"<extra></extra>"
            ),
        )
    )

fig.update_layout(
    title="Flamechart equivalent (synthetic data — Linaro MAP 25.1.2 format)",
    xaxis_title="Cumulative CPU Time (%)",
    yaxis_title="Hotspot Rank",
    barmode="relative",
    paper_bgcolor="#000000",
    plot_bgcolor="#111111",
    font=dict(color="white", size=12),
    hovermode="closest",
    height=500,
    showlegend=False,
)

fig.update_yaxes(autorange="reversed")

fig.show()

## Optimization Analysis

In [ ]:
recommendations = analyze_profile_for_optimization(profile)

print("\n=== Optimization Recommendations ===")
print(f"Total recommendations: {len(recommendations)}\n")

for i, rec in enumerate(recommendations, 1):
    print(f"{i}. [{rec['priority']}] {rec['category']}")
    print(f"   {rec['suggestion']}")
    print(f"   Metric: {rec['metric_value']:.1f}%\n")

## Recommendations as Styled DataFrame

In [ ]:
rec_df = pd.DataFrame(recommendations)

def priority_color(val):
    """Color rows by priority."""
    if val == "HIGH":
        return "background-color: #CC0000; color: white;"
    elif val == "MEDIUM":
        return "background-color: #FFAA00; color: white;"
    else:
        return "background-color: #0066CC; color: white;"

styled_rec = rec_df.style.applymap(
    priority_color, subset=["priority"]
)

display(styled_rec)

## How to Launch MAP on a Neoverse Cluster Under Slurm

**Prerequisites:**
- Linaro Forge installed and licensed (or in trial period).
- Application compiled with debug symbols or `−g` flag for source attribution.
- Slurm job submission with `srun`.

**Basic usage:**
```bash
map --profile srun ./hpl_benchmark
```
(facts — from verified MAP documentation)

This launches `./hpl_benchmark` under MAP's profiler wrapper. Output: `hpl_benchmark_8p_2026-04-10_14-23.map`.

**Post-run analysis:**
```bash
perf-report hpl_benchmark_8p_47.3s.map
```
(facts — from verified MAP documentation)

This generates an HTML or text summary of the profiling results.

**Neoverse V3/N3 support:**
Neoverse V3 and N3 explicit MAP support is not yet confirmed in public Linaro Forge release notes as of 2026-04-10. However, given SPE (Sampling Profiling Extension) availability on Neoverse platforms and Linaro's commitment to Arm server silicon, support is likely. (speculative)

**Tips:**
- Use `map --help` for all options (e.g., `--sampling-interval`, `--pause-on-error`).
- For weak scaling studies, profile once per node count to see how communication scales.
- MAP overhead (~5%) is small enough to use in production batch jobs. (facts)

## Summary

**What we learned:**

1. **Linaro Forge MAP is the only MPI-aware profiler** that combines low overhead (~5%), source attribution, and rank-level profiling. Commercial tool, 2-week free trial. (facts)

2. **Time breakdown analysis** reveals where runtime is spent: CPU (compute), MPI (communication), I/O (storage), thread overhead (synchronization).

3. **Hotspots table** identifies the top functions consuming time. The top 3–5 are the best targets for optimization.

4. **Flamechart visualization** (horizontal bar chart) shows CPU time distribution across hotspots, colored by category.

5. **Optimization recommendations** apply heuristic rules:
   - MPI > 20% → use non-blocking collectives or overlap with computation
   - I/O > 10% → use MPI-IO or buffering
   - Compute hotspot > 30% → vectorize with SVE2 or call ArmPL BLAS

**Next steps:**
- Download a 2-week trial of Linaro Forge MAP from [linaroforge.com](https://www.linaroforge.com).
- Profile your own HPC application on a Neoverse cluster.
- Use the flamechart and recommendations to guide optimization.
- Rerun MAP after each optimization to verify improvement.

---

**References:**
- Linaro Forge 25.1.2 Documentation: https://docs.linaroforge.com/25.1.2/html/forge/forge/introduction_to_forge/map.html
- NERSC MAP Guide: https://docs.nersc.gov/tools/performance/map/
- Linaro newsroom (Arm Forge acquisition): https://www.linaro.org/news/linaro-to-acquire-arm-forge-software-tools-business/
- ARM SW Stack Series Post 2 research corpus: `/ARM_SW_Stack/Research/verified/arm-forge-map.md`